In [45]:
# GPU check
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

!pip install -q transformers datasets scikit-learn tqdm


CUDA: True
Tesla T4


In [46]:
from google.colab import files
import polars as pl

print("📤 Upload ton fichier CSV:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_full = pl.read_csv(filename)

print(f"\n📊 Dataset: {len(df_full)} lignes")
print(f"Colonnes: {df_full.columns}")

required_cols = ["text_clean", "label", "label_id"]
missing = [c for c in required_cols if c not in df_full.columns]
if missing:
    raise ValueError(f"Colonnes manquantes: {missing}")

print("\n📊 Distribution labels:")
print(df_full.group_by("label").len().sort("label"))


📤 Upload ton fichier CSV:


Saving test.csv to test (2).csv

📊 Dataset: 42000 lignes
Colonnes: ['text_clean', 'label', 'label_id']

📊 Distribution labels:
shape: (7, 2)
┌──────────┬──────┐
│ label    ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ anger    ┆ 6000 │
│ fear     ┆ 6000 │
│ joy      ┆ 6000 │
│ love     ┆ 6000 │
│ neutral  ┆ 6000 │
│ sad      ┆ 6000 │
│ surprise ┆ 6000 │
└──────────┴──────┘


In [47]:
# =============================
# 2️⃣ Convertir en pandas + HF Dataset
# =============================
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

df = df_full.to_pandas()

# Créer jeu train/validation
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df['label_id'], random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


In [48]:
# =============================
# 3️⃣ Data augmentation : créer short texts artificiels
# =============================
import random

def create_short_text(text, min_words=3, max_words=15):
    words = text.split()
    if len(words) <= max_words:
        return text
    start = random.randint(0, len(words) - max_words)
    end = start + random.randint(min_words, max_words)
    return " ".join(words[start:end])

# Dupliquer dataset pour inclure short texts
short_texts = []
for i, row in train_df.iterrows():
    short_version = create_short_text(row['text_clean'])
    short_texts.append({"text_clean": short_version, "label": row['label'], "label_id": row['label_id']})

# Ajouter à train dataset
train_df_aug = pd.concat([train_df, pd.DataFrame(short_texts)], ignore_index=True)
train_dataset = Dataset.from_pandas(train_df_aug)

print(f"\n📊 Taille dataset train après ajout short texts: {len(train_dataset)}")



📊 Taille dataset train après ajout short texts: 75600


In [56]:
# =============================
# 4️⃣ Tokenizer
# =============================
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128  # pour court texte
CHUNK_SIZE = 128  # pour long texte

def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/75600 [00:00<?, ? examples/s]

Map:   0%|          | 0/4200 [00:00<?, ? examples/s]

In [57]:
# =============================
# 5️⃣ PyTorch Dataset
# =============================
import torch

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            'input_ids': torch.tensor(item['input_ids']),
            'attention_mask': torch.tensor(item['attention_mask']),
            'labels': torch.tensor(item['label_id'])
        }

train_data = EmotionDataset(train_dataset)
val_data = EmotionDataset(val_dataset)

In [58]:
# =============================
# 6️⃣ Model
# =============================
from transformers import AutoModelForSequenceClassification

NUM_LABELS = len(df['label_id'].unique())
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [59]:
!pip install --upgrade transformers -q

# =============================
# 7️⃣ Training
# =============================
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    do_train=True,
    do_eval=True,
    # dans les anciennes versions, il n'y a pas evaluation_strategy ni save_strategy
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [60]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [61]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss
50,1.938374
100,1.582791
150,1.186030
200,1.044978
250,0.902697
300,0.844294
350,0.797850
400,0.758828
450,0.704458
500,0.716943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14175, training_loss=0.4597953561920758, metrics={'train_runtime': 5121.0332, 'train_samples_per_second': 44.288, 'train_steps_per_second': 2.768, 'total_flos': 1.491906657024e+16, 'train_loss': 0.4597953561920758, 'epoch': 3.0})

In [68]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # important pour l'inference



RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [69]:
def predict_emotion_single(text, model, tokenizer, max_len=128):
    """
    Prédit l'émotion d'un texte court ou long.
    """
    # Préparer l'input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len
    )

    # Envoyer sur le même device que le modèle
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Pas besoin de calcul de gradient pour l'inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_id = logits.argmax(-1).item()  # id de la classe prédite

    # Convertir l'id en label
    label_map = df[['label', 'label_id']].drop_duplicates().set_index('label_id')['label'].to_dict()
    return label_map[pred_id]



In [70]:
example_text = "I am happy that I did it"
predicted_emotion = predict_emotion_single(example_text, model, tokenizer)
print("Texte:", example_text)
print("Emotion prédite:", predicted_emotion)


Texte: I am happy that I did it
Emotion prédite: joy


In [71]:
# Liste de textes à tester (courts et longs)
test_texts = [
    "I am happy that I did it",  # court - joy
    "I love spending time with my family",  # court - love
    "I hate when people lie to me",  # court - hate
    "It's just another normal day",  # court - neutral
    "She is sad because she lost her favorite book",  # court - sadness
    "I feel anxious before every presentation",  # court - fear
    "I can't stop thinking about that betrayal",  # court - anger/hate
    "He is proud of his accomplishments",  # court - joy
    "The sunset over the mountains fills me with awe and love for nature",  # long - love
    "After months of hard work, seeing the results brought tears of joy and relief",  # long - joy
    "The constant noise and chaos in the city makes me stressed and frustrated",  # long - anger/fear
    "Losing someone you care about leaves a hollow feeling of sadness",  # long - sadness
    "Watching the baby take its first steps filled me with happiness",  # long - joy
    "She was terrified when she realized she was lost in the dark forest",  # long - fear
    "He felt a mix of hate and betrayal after being lied to by his closest friend"  # long - hate
]

# Boucle pour prédire l'émotion de chaque texte
for text in test_texts:
    predicted_emotion = predict_emotion_single(text, model, tokenizer)
    print(f"Texte: {text}")
    print(f"Emotion prédite: {predicted_emotion}")
    print("-" * 60)


Texte: I am happy that I did it
Emotion prédite: joy
------------------------------------------------------------
Texte: I love spending time with my family
Emotion prédite: joy
------------------------------------------------------------
Texte: I hate when people lie to me
Emotion prédite: anger
------------------------------------------------------------
Texte: It's just another normal day
Emotion prédite: neutral
------------------------------------------------------------
Texte: She is sad because she lost her favorite book
Emotion prédite: sad
------------------------------------------------------------
Texte: I feel anxious before every presentation
Emotion prédite: fear
------------------------------------------------------------
Texte: I can't stop thinking about that betrayal
Emotion prédite: anger
------------------------------------------------------------
Texte: He is proud of his accomplishments
Emotion prédite: joy
---------------------------------------------------------

In [72]:
# Crée un dossier pour le modèle
save_path = "./emotion_model_roberta_base_en"

# Sauvegarder le modèle fine-tuné
model.save_pretrained(save_path)

# Sauvegarder le tokenizer
tokenizer.save_pretrained(save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./emotion_model_roberta_base_en/tokenizer_config.json',
 './emotion_model_roberta_base_en/tokenizer.json')

In [73]:
import pickle

# Dossier où tu veux sauvegarder ton pickle
save_file = "emotion_model_roberta_base_en.pkl"

# Créer un dictionnaire contenant modèle + tokenizer
model_data = {
    "model": model,
    "tokenizer": tokenizer
}

# Sauvegarder tout dans un pickle
with open(save_file, "wb") as f:
    pickle.dump(model_data, f)


recharger le model

In [ ]:
import pickle
import torch

# Charger le fichier
with open("emotion_model.pkl", "rb") as f:
    model_data = pickle.load(f)

model = model_data["model"]
tokenizer = model_data["tokenizer"]

# Mettre le modèle sur GPU si dispo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
